# Week 14: PyTorch, training loops, and debugging training

| Session | Topic | Time |
|---|---|---|
| Day 1 | Tensors and autograd: PyTorch checks your Week 13 gradients | 2 h |
| Day 2 | `nn.Module`, `Dataset`/`DataLoader`, a reusable training loop | 3 h |
| Day 3 | Diagnostics: overfit one batch, the learning-rate range test, loss curves | 2.5 h |
| Day 4 | Debugging: fix a training script with 6 planted bugs; seeds | 2 h |
| Review | Spaced recall | 0.5 h |

**Install** (once, CPU build, about 200 MB): `pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu`
**Read:** the PyTorch tutorials *Learn the Basics* (Tensors, Datasets & DataLoaders, Build the Neural Network, Autograd, Optimization), and Karpathy's blog post *A Recipe for Training Neural Networks* (essential; you'll come back to it for the rest of the course).

### Done when
- [ ] Every test cell prints ✅.
- [ ] You can write a train/eval loop from a blank page, including `model.train()`/`model.eval()`, `zero_grad`, and `torch.no_grad()`.
- [ ] You found all 6 bugs on Day 4 and can explain the symptom each one causes.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

---
# Day 1: Tensors and autograd (2 h)

A **tensor** is a NumPy array that can (1) live on a GPU and (2) record operations for automatic differentiation. Set `requires_grad=True` on a tensor and every operation on it builds a graph, exactly like your Week 13 `Value` class. `loss.backward()` fills `.grad`.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(a.shape, a.dtype, a @ a, a.numpy())          # NumPy-like API
w = torch.tensor([2.0, -1.0], requires_grad=True)
loss = ((a @ w) ** 2).sum()
loss.backward()
print("autograd:", w.grad, "| by hand: 2·aᵀ(a w) =", 2 * a.T @ (a @ w.detach()))

### Exercise 14.1: PyTorch checks your Week 13 backprop
1. Paste your Week 13 `relu`, `relu_grad`, `softmax`, `forward` and `backward` into the first cell.
2. Write `torch_loss(params_t, X, Y)`, the same 2-layer MLP loss in PyTorch (`torch.relu`, `torch.log_softmax`).
3. The test compares your hand-written gradients with PyTorch's autograd.

In [ ]:
# 1. paste your Week 13 functions here

In [ ]:
def torch_loss(params_t, X, Y):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
Xn = rng.normal(size=(16, 5)); Yn = np.eye(3)[rng.integers(0, 3, 16)]
p = {"W1": 0.5 * rng.normal(size=(5, 8)), "b1": 0.1 * rng.normal(size=8), "W2": 0.5 * rng.normal(size=(8, 3)), "b2": 0.1 * rng.normal(size=3)}
loss_np, cache = forward(p, Xn, Yn); g_np = backward(p, cache)
pt = {k: torch.tensor(v, requires_grad=True) for k, v in p.items()}
loss_t = torch_loss(pt, torch.tensor(Xn), torch.tensor(Yn)); loss_t.backward()
assert abs(loss_t.item() - loss_np) < 1e-6, (loss_t.item(), loss_np)
for k in p:
    assert np.allclose(pt[k].grad.numpy(), g_np[k], atol=1e-6), k
print("✅ 14.1 passed: your hand-derived backprop equals PyTorch autograd")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`Z1 = X @ W1 + b1`, `H = torch.relu(Z1)`, `logp = torch.log_softmax(H @ W2 + b2, dim=1)`, and the loss is `-(Y * logp).sum(dim=1).mean()`.

</details>


---
# Day 2: The standard PyTorch workflow (3 h)

```
Dataset  → gives (x, y) for an index
DataLoader → batches + shuffles
nn.Module → the model: layers in __init__, computation in forward()
loss_fn, optimizer → nn.CrossEntropyLoss(), torch.optim.AdamW(model.parameters(), lr)
loop: for each batch → zero_grad → forward → loss → backward → step
```
**Two modes:** `model.train()` turns on dropout and batch-norm updates; `model.eval()` turns them off. Evaluate inside `with torch.no_grad():` (no graph, so it's faster and uses less memory).

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

Xb, yb = load_breast_cancer(return_X_y=True)
X_tr, X_va, y_tr, y_va = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=0)
mu, sd = X_tr.mean(0), X_tr.std(0)                     # training statistics only (Week 8)
to_ds = lambda X, y: TensorDataset(torch.tensor((X - mu) / sd, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
train_ds, val_ds = to_ds(X_tr, y_tr), to_ds(X_va, y_va)

class MLP(nn.Module):
    def __init__(self, d_in, d_hidden=64, n_classes=2, p_drop=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(d_hidden, n_classes))

    def forward(self, x):
        return self.net(x)          # raw scores ("logits"); CrossEntropyLoss applies log-softmax itself

print(MLP(30)); print("parameters:", sum(p.numel() for p in MLP(30).parameters()))

### Exercise 14.2: A reusable training loop
Write two functions that you'll reuse for the rest of the course (and later move into your capstone):
- `train_one_epoch(model, loader, loss_fn, optimizer, device="cpu") -> mean training loss`
- `evaluate(model, loader, loss_fn, device="cpu") -> (mean loss, accuracy)`

Weight each batch's loss by its size when you average. Then train `MLP(30)` for 30 epochs with `AdamW(lr=1e-3)` and batch size 32.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device="cpu"):
    raise NotImplementedError

def evaluate(model, loader, loss_fn, device="cpu"):
    raise NotImplementedError

In [ ]:
set_seed(0)
model = MLP(30); loss_fn = nn.CrossEntropyLoss(); opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True); val_dl = DataLoader(val_ds, batch_size=256)
hist = []
for epoch in range(30):
    tr_loss = train_one_epoch(model, train_dl, loss_fn, opt)
    va_loss, va_acc = evaluate(model, val_dl, loss_fn)
    hist.append((tr_loss, va_loss, va_acc))
print(f"final val acc {va_acc:.3f}")
plt.plot([h[0] for h in hist], label="train loss"); plt.plot([h[1] for h in hist], label="val loss"); plt.legend(); plt.show()
assert va_acc > 0.95
l1 = evaluate(model, val_dl, loss_fn); l2 = evaluate(model, val_dl, loss_fn)
assert l1 == l2, "evaluate is not deterministic: did you call model.eval()?"
print("✅ 14.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`train_one_epoch`: `model.train()`. For each batch: `x, y = x.to(device), y.to(device)`; `optimizer.zero_grad()`; `loss = loss_fn(model(x), y)`; `loss.backward()`; `optimizer.step()`; accumulate `loss.item() * len(x)`.

</details>

<details><summary>Hint 2</summary>

`evaluate`: `model.eval()` and `with torch.no_grad():`. Accuracy is `(logits.argmax(1) == y).sum().item()` accumulated, divided by the dataset size.

</details>


---
# Day 3: Training diagnostics (2.5 h)

From Karpathy's recipe, the checks that catch most problems:
1. **Check the loss at initialisation.** For C balanced classes it should be about $\log C$ (Week 7: a uniform prediction).
2. **Overfit a single batch.** A correct model + loop must drive the loss on one fixed batch of 16 to ≈ 0. If it can't, there's a bug, and more data won't help.
3. **Find a learning rate** with a *range test*: increase the LR exponentially, batch by batch, and plot the loss. Pick a value about 10× below where the loss is lowest, before it explodes.
4. **Watch both curves.** Train ↓ with val ↑ means overfitting. Both flat and high means underfitting or a bug. Spikes mean the LR is too high.

In [ ]:
set_seed(0); m = MLP(30, p_drop=0.0)
xb, yb_ = next(iter(DataLoader(train_ds, batch_size=16, shuffle=True)))
print("loss at init:", nn.CrossEntropyLoss()(m(xb), yb_).item(), "≈ log(2) =", np.log(2))
opt = torch.optim.AdamW(m.parameters(), lr=1e-2)
for step in range(300):
    opt.zero_grad(); l = nn.CrossEntropyLoss()(m(xb), yb_); l.backward(); opt.step()
print("loss after overfitting one batch:", l.item())

### Exercise 14.3: `lr_range_test`
For each learning rate in `lrs` (increasing), take **one** optimisation step on the next training batch (cycle through the loader as needed), with `opt.param_groups[0]['lr'] = lr`, and record the loss *before* the step. Use a fresh model from `model_fn()` and AdamW. Stop early if the loss exceeds 4× the smallest loss so far. Return the list of losses.

In [ ]:
def lr_range_test(model_fn, loader, lrs):
    raise NotImplementedError

In [ ]:
set_seed(0)
lrs = np.logspace(-5, 1, 80)
losses = lr_range_test(lambda: MLP(30), DataLoader(train_ds, batch_size=32, shuffle=True), lrs)
assert 10 <= len(losses) <= len(lrs)
plt.semilogx(lrs[:len(losses)], losses); plt.xlabel("learning rate"); plt.ylabel("loss"); plt.title("LR range test"); plt.show()
print("✅ 14.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Use `it = iter(loader)` and `try: x, y = next(it)` / `except StopIteration: it = iter(loader); x, y = next(it)`.

</details>


✍️ *Which learning rate would you pick from the plot, and why?* (double-click to edit)



---
# Day 4: Debug a broken training script (2 h)

The function below trains on the breast cancer data and reports a validation accuracy. It contains **6 bugs**. Some crash, some quietly make results wrong, and some make them impossible to reproduce. Fix all 6, using the diagnostics from Day 3. Next to each fix, write a comment `# BUG n: …` saying what the symptom was.

In [ ]:
def buggy_train(seed=0, epochs=40):
    model = MLP(30, p_drop=0.3)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.05)
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_dl = DataLoader(train_ds, batch_size=256)
    for epoch in range(epochs):
        model.train()
        for x, y in train_dl:
            probs = torch.softmax(model(x), dim=1)
            loss = loss_fn(probs, y.float())
            loss.backward()
            opt.step()
    correct = 0
    for x, y in val_dl:
        correct += (model(x).argmax(1) == y).sum().item()
    return model, correct / len(val_dl.dataset)

In [ ]:
m1, acc1 = buggy_train(seed=0); m2, acc2 = buggy_train(seed=0)
assert acc1 == acc2, "not reproducible: same seed, different result"
val_dl = DataLoader(val_ds, batch_size=256)
m1.eval()
with torch.no_grad():
    true_acc = sum((m1(x).argmax(1) == y).sum().item() for x, y in val_dl) / len(val_ds)
assert abs(true_acc - acc1) < 1e-9, "the reported accuracy isn't the validation accuracy"
assert true_acc > 0.94, f"validation accuracy {true_acc:.3f} is too low; a bug is still slowing training"
import inspect; src = inspect.getsource(buggy_train)
assert "zero_grad" in src and "no_grad" in src and src.count("# BUG") >= 6
print(f"✅ 14.4 passed: val acc {true_acc:.3f}, reproducible")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Run the cell and read the crash first. Then: should `CrossEntropyLoss` receive probabilities or logits? What dtype should class labels be?

</details>

<details><summary>Hint 2</summary>

Which data is the 'validation' loader built from? What does `seed` actually control in this function?

</details>

<details><summary>Hint 3</summary>

What happens to `.grad` between steps if nobody resets it? Is dropout active during evaluation?

</details>


---
# Review (30 min)
**Blank page:** write `train_one_epoch` and `evaluate` from memory.
1. *(Week 8)* Why fit the scaler on training data only?
2. *(Week 10)* What does nDCG reward that P@k ignores?
3. *(Week 13)* What does `loss.backward()` do, step by step?
4. Why must you call `zero_grad()`?
5. What do `model.eval()` and `torch.no_grad()` each do? Are they the same?
6. Why give `CrossEntropyLoss` logits rather than probabilities?
7. What's the expected loss at initialisation for 10 balanced classes?

✍️ *Your answer:* (double-click to edit)

